## Statistical Hypothesis Testing

This section evaluates statistically significant relationships
in the Online Retail dataset to support evidence-based
business decisions.

The analysis will include:
- Research questions and hypotheses
- Selection of appropriate statistical tests
- Significance level (α = 0.05)
- Test results and p-values
- Business interpretation and recommendations

### Hypothesis 1: Customer Spending and Repeat Purchases

**Research Question:** Do repeat customers spend more than
one-time customers?

**H₀ (Null Hypothesis):**
The mean total spending of repeat customers is equal to
the mean total spending of one-time customers.

**H₁ (Alternative Hypothesis):**
The mean total spending differs between repeat customers
and one-time customers.

**Significance Level:** α = 0.05

**Unit of Analysis:** Individual customers.

**Planned Test:** Welch's independent samples t-test,
subject to data validation.

**Business Relevance:**
Understanding spending differences between repeat and
one-time customers can help inform customer retention
and marketing strategies.

In [1]:
# STEP 1: Load customer intelligence data

import pandas as pd
import numpy as np
from pathlib import Path

# Locate the project directory
project_root = Path.cwd().resolve()

if project_root.name == "jupyter_notebooks":
    project_root = project_root.parent

# Load customer-level data
customer_path = (
    project_root
    / "data"
    / "tableau"
    / "retail_customer_intelligence.csv"
)

customer_df = pd.read_csv(customer_path)

print("Customer dataset shape:", customer_df.shape)

# Inspect available columns
print("\nColumn names:")
print(customer_df.columns.tolist())

# Preview first five rows
display(customer_df.head())

Customer dataset shape: (4337, 10)

Column names:
['CustomerID', 'TotalSpending', 'TotalOrders', 'TotalUnits', 'FirstPurchase', 'LastPurchase', 'PurchaseFrequency', 'AverageOrderValue', 'CustomerType', 'RecencyDays']


,CustomerID,TotalSpending,TotalOrders,TotalUnits,FirstPurchase,LastPurchase,PurchaseFrequency,AverageOrderValue,CustomerType,RecencyDays
0,12346,77183.60,1,74215,2011-01-18 10:01:00,2011-01-18 10:01:00,1,77183.600000,One-Time Customer,326
1,12347,4310.00,7,2458,2010-12-07 14:57:00,2011-12-07 15:52:00,7,615.714286,Repeat Customer,3
2,12348,1797.24,4,2341,2010-12-16 19:09:00,2011-09-25 13:13:00,4,449.310000,Repeat Customer,76
3,12349,1757.55,1,631,2011-11-21 09:51:00,2011-11-21 09:51:00,1,1757.550000,One-Time Customer,19
4,12350,334.40,1,197,2011-02-02 16:01:00,2011-02-02 16:01:00,1,334.400000,One-Time Customer,311


In [2]:
# STEP 2: Validate customer data for hypothesis testing

# Check missing values in required columns
print("Missing values:")
print(
    customer_df[
        ["CustomerID", "TotalSpending", "TotalOrders", "CustomerType"]
    ].isnull().sum()
)

# Check duplicate customer IDs
print("\nDuplicate customer IDs:")
print(customer_df["CustomerID"].duplicated().sum())

# Check invalid spending values
print("\nCustomers with zero or negative spending:")
print((customer_df["TotalSpending"] <= 0).sum())

# Check customer group sizes
print("\nCustomer type distribution:")
print(customer_df["CustomerType"].value_counts())

# Verify customer classification
classification_errors = (
    (
        (customer_df["CustomerType"] == "One-Time Customer")
        & (customer_df["TotalOrders"] != 1)
    )
    |
    (
        (customer_df["CustomerType"] == "Repeat Customer")
        & (customer_df["TotalOrders"] <= 1)
    )
)

print("\nCustomer classification inconsistencies:")
print(classification_errors.sum())

Missing values:
CustomerID       0
TotalSpending    0
TotalOrders      0
CustomerType     0
dtype: int64

Duplicate customer IDs:
0

Customers with zero or negative spending:
0

Customer type distribution:
CustomerType
Repeat Customer      2844
One-Time Customer    1493
Name: count, dtype: int64

Customer classification inconsistencies:
0


In [3]:
# STEP 3: Prepare customer groups for hypothesis testing

# Extract total spending for repeat customers
repeat_spending = customer_df.loc[
    customer_df["CustomerType"] == "Repeat Customer",
    "TotalSpending"
]

# Extract total spending for one-time customers
one_time_spending = customer_df.loc[
    customer_df["CustomerType"] == "One-Time Customer",
    "TotalSpending"
]

# Display descriptive statistics for each group
print("REPEAT CUSTOMERS")
print(repeat_spending.describe())

print("\nONE-TIME CUSTOMERS")
print(one_time_spending.describe())

# Compare average spending
print("\nAVERAGE CUSTOMER SPENDING")
print(f"Repeat customers: £{repeat_spending.mean():,.2f}")
print(f"One-time customers: £{one_time_spending.mean():,.2f}")

# Compare median spending
print("\nMEDIAN CUSTOMER SPENDING")
print(f"Repeat customers: £{repeat_spending.median():,.2f}")
print(f"One-time customers: £{one_time_spending.median():,.2f}")

REPEAT CUSTOMERS
count      2844.000000
mean       2908.837076
std       10901.360364
min           6.900000
25%         615.777500
50%        1155.535000
75%        2433.850000
max      280206.020000
Name: TotalSpending, dtype: float64

ONE-TIME CUSTOMERS
count     1493.000000
mean       411.245520
std       2034.246264
min          3.750000
25%        152.650000
50%        255.910000
75%        406.090000
max      77183.600000
Name: TotalSpending, dtype: float64

AVERAGE CUSTOMER SPENDING
Repeat customers: £2,908.84
One-time customers: £411.25

MEDIAN CUSTOMER SPENDING
Repeat customers: £1,155.53
One-time customers: £255.91


In [4]:
# STEP 4: Examine spending skewness and potential outliers

import pandas as pd

for group_name, spending in [
    ("Repeat Customers", repeat_spending),
    ("One-Time Customers", one_time_spending)
]:
    q1 = spending.quantile(0.25)
    q3 = spending.quantile(0.75)
    iqr = q3 - q1

    upper_bound = q3 + 1.5 * iqr
    outlier_count = (spending > upper_bound).sum()

    print(f"\n{group_name.upper()}")
    print(f"Skewness: {spending.skew():.2f}")
    print(f"IQR: £{iqr:,.2f}")
    print(f"Upper outlier boundary: £{upper_bound:,.2f}")
    print(f"Potential high-spending outliers: {outlier_count:,}")
    print(f"Outlier percentage: {outlier_count / len(spending) * 100:.2f}%")


REPEAT CUSTOMERS
Skewness: 16.17
IQR: £1,818.07
Upper outlier boundary: £5,160.96
Potential high-spending outliers: 256
Outlier percentage: 9.00%

ONE-TIME CUSTOMERS
Skewness: 36.13
IQR: £253.44
Upper outlier boundary: £786.25
Potential high-spending outliers: 105
Outlier percentage: 7.03%


In [5]:
import scipy
from scipy import stats

print("SciPy version:", scipy.__version__)
print("SciPy imported successfully!")

SciPy version: 1.13.1
SciPy imported successfully!


In [6]:
# STEP 5: Prepare customer spending groups for hypothesis testing

repeat_spending = customer_df.loc[
    customer_df["CustomerType"] == "Repeat Customer",
    "TotalSpending"
].dropna()

one_time_spending = customer_df.loc[
    customer_df["CustomerType"] == "One-Time Customer",
    "TotalSpending"
].dropna()

print("CUSTOMER GROUPS FOR HYPOTHESIS TESTING")
print(f"Repeat customers: {len(repeat_spending):,}")
print(f"One-time customers: {len(one_time_spending):,}")

print("\nAVERAGE SPENDING")
print(f"Repeat customers: £{repeat_spending.mean():,.2f}")
print(f"One-time customers: £{one_time_spending.mean():,.2f}")

CUSTOMER GROUPS FOR HYPOTHESIS TESTING
Repeat customers: 2,844
One-time customers: 1,493

AVERAGE SPENDING
Repeat customers: £2,908.84
One-time customers: £411.25


In [7]:
# STEP 6: Welch's Independent-Samples T-Test

# Perform a two-sided Welch's t-test
t_statistic, p_value = stats.ttest_ind(
    repeat_spending,
    one_time_spending,
    equal_var=False
)

# Set significance level
alpha = 0.05

print("WELCH'S INDEPENDENT-SAMPLES T-TEST")
print("-" * 45)

print(f"T-statistic: {t_statistic:.4f}")
print(f"P-value: {p_value:.6g}")
print(f"Significance level: {alpha}")

print("\nHYPOTHESIS TEST DECISION")

if p_value < alpha:
    print("Reject the null hypothesis (H0).")
    print("There is a statistically significant difference")
    print("in mean spending between the two customer groups.")
else:
    print("Fail to reject the null hypothesis (H0).")
    print("There is insufficient evidence of a difference")
    print("in mean spending between the two customer groups.")

WELCH'S INDEPENDENT-SAMPLES T-TEST
---------------------------------------------
T-statistic: 11.8320
P-value: 1.20273e-31
Significance level: 0.05

HYPOTHESIS TEST DECISION
Reject the null hypothesis (H0).
There is a statistically significant difference
in mean spending between the two customer groups.


In [8]:
# STEP 7: Mann-Whitney U Test

# Compare repeat and one-time customer spending
u_statistic, mw_p_value = stats.mannwhitneyu(
    repeat_spending,
    one_time_spending,
    alternative="two-sided"
)

alpha = 0.05

print("MANN-WHITNEY U TEST")
print("-" * 45)

print(f"U-statistic: {u_statistic:,.2f}")
print(f"P-value: {mw_p_value:.6g}")
print(f"Significance level: {alpha}")

print("\nHYPOTHESIS TEST DECISION")

if mw_p_value < alpha:
    print("Reject the null hypothesis (H0).")
    print("There is a statistically significant difference")
    print("in the spending distributions of the two groups.")
else:
    print("Fail to reject the null hypothesis (H0).")
    print("There is insufficient evidence of a difference")
    print("in the spending distributions of the two groups.")

MANN-WHITNEY U TEST
---------------------------------------------
U-statistic: 3,787,067.00
P-value: 0
Significance level: 0.05

HYPOTHESIS TEST DECISION
Reject the null hypothesis (H0).
There is a statistically significant difference
in the spending distributions of the two groups.


### Step 8: Hypothesis Testing Results and Business Interpretation

**Research Question:** Do repeat customers spend more than one-time customers?

**Statistical Results:**

- Welch's independent-samples t-test: t = 11.8320, p ≈ 1.20 × 10⁻³¹.
- Mann–Whitney U test: U = 3,787,067, p extremely small (reported as 0 due to numerical precision).
- Significance level: α = 0.05.

**Descriptive Findings:**

- Repeat customers (n = 2,844): Mean spending = £2,908.84; median = £1,155.53.
- One-time customers (n = 1,493): Mean spending = £411.25; median = £255.91.
- Difference in mean spending: approximately £2,497.59.

**Conclusion:**

Both statistical tests reject their respective null hypotheses. Repeat customers demonstrate significantly higher average spending, and the spending distributions differ significantly between the two groups.

**Business Implications:**

The retailer should consider targeted retention campaigns, personalised offers, and loyalty initiatives to encourage repeat purchases and strengthen customer relationships.

**Limitations:**

Customer spending is highly right-skewed and contains substantial outliers. Welch's t-test compares means, while the Mann–Whitney U test compares distributions. The findings demonstrate statistical associations rather than causal effects. Furthermore, repeat customers have more purchasing opportunities by definition, which contributes to their higher cumulative spending.

## Hypothesis Test 2: Purchase Frequency and Average Order Value

**Research Question:** Is there a statistically significant association between customer purchase frequency and average order value?

**H₀ (Null Hypothesis):** There is no monotonic association between purchase frequency and average order value.

**H₁ (Alternative Hypothesis):** There is a monotonic association between purchase frequency and average order value.

**Significance Level:** α = 0.05

**Unit of Analysis:** Individual customers.

**Planned Test:** Spearman's rank correlation.

**Business Relevance:** Understanding whether frequent purchasers also place higher-value orders can help inform customer segmentation, loyalty initiatives, and marketing strategies.

In [11]:
# STEP 11: Validate variables for Hypothesis Test 2

# Select the variables required for correlation analysis
correlation_data = customer_df[
    ["PurchaseFrequency", "AverageOrderValue"]
].copy()

# Check missing values
print("MISSING VALUES")
print(correlation_data.isnull().sum())

# Check dataset size
print("\nDataset shape:", correlation_data.shape)

# Display descriptive statistics
print("\nDESCRIPTIVE STATISTICS")
display(correlation_data.describe())

MISSING VALUES
PurchaseFrequency    0
AverageOrderValue    0
dtype: int64

Dataset shape: (4337, 2)

DESCRIPTIVE STATISTICS


,PurchaseFrequency,AverageOrderValue
count,4337.000000,4337.000000
mean,4.272539,417.685924
std,7.698808,1796.716543
min,1.000000,3.450000
25%,1.000000,177.865000
50%,2.000000,292.000000
75%,5.000000,428.300000
max,209.000000,84236.250000


In [12]:

# STEP 12: Spearman's Rank Correlation Test

# Perform Spearman's rank correlation
spearman_rho, spearman_p_value = stats.spearmanr(
    correlation_data["PurchaseFrequency"],
    correlation_data["AverageOrderValue"]
)

alpha = 0.05

print("SPEARMAN'S RANK CORRELATION TEST")
print("-" * 45)

print(f"Correlation coefficient (rho): {spearman_rho:.4f}")
print(f"P-value: {spearman_p_value:.6g}")
print(f"Significance level: {alpha}")

print("\nHYPOTHESIS TEST DECISION")

if spearman_p_value < alpha:
    print("Reject the null hypothesis (H0).")
    print("There is a statistically significant")
    print("monotonic association between purchase")
    print("frequency and average order value.")
else:
    print("Fail to reject the null hypothesis (H0).")
    print("There is insufficient evidence of a")
    print("monotonic association between the variables.")


SPEARMAN'S RANK CORRELATION TEST
---------------------------------------------
Correlation coefficient (rho): 0.1563
P-value: 4.06594e-25
Significance level: 0.05

HYPOTHESIS TEST DECISION
Reject the null hypothesis (H0).
There is a statistically significant
monotonic association between purchase
frequency and average order value.


### Hypothesis Test 2: Results and Business Interpretation

**Statistical Results**

- **Test:** Spearman's Rank Correlation
- **Sample Size:** 4,337 customers
- **Correlation Coefficient (ρ):** 0.1563
- **P-value:** 4.06594 × 10⁻²⁵
- **Significance Level:** α = 0.05

**Hypothesis Decision**

Reject the null hypothesis (H₀). There is a statistically significant positive monotonic association between customer purchase frequency and average order value.

**Business Interpretation**

The correlation is weak (ρ = 0.1563), indicating that customers who purchase more frequently tend to have slightly higher average order values. However, purchase frequency alone does not strongly explain differences in order value.

**Business Recommendations**

The retailer should combine customer purchase frequency with monetary value and recency when developing customer segments. Personalised promotions, loyalty programmes, and targeted cross-selling may help increase engagement and spending, but their effectiveness should be evaluated through further analysis or experimentation.

**Limitations**

The analysis is observational and does not establish causation. Extreme average order values and differences in customer purchasing behaviour may influence the results. Statistical significance should not be confused with the practical strength of the relationship.

## Hypothesis Test 3: UK vs International Order Values

**Research Question:** Is there a statistically significant difference in mean order value between UK and international customers' orders?

**H₀ (Null Hypothesis):** The mean order value is equal for UK and international orders.

**H₁ (Alternative Hypothesis):** The mean order value differs between UK and international orders.

**Significance Level:** α = 0.05

**Unit of Analysis:** Individual invoices (orders).

**Planned Test:** Welch's independent-samples t-test, with a Mann–Whitney U test as a complementary non-parametric analysis if appropriate.

**Business Relevance:** Understanding geographical differences in order values can support international market development, targeted marketing and revenue diversification.

**Methodological Note:** We will aggregate sales to invoice level before testing, validate geographical classifications and inspect the spending distributions. Statistical findings will not establish that customer location causes differences in order value.

In [13]:

# STEP 16: Load sales data for Hypothesis Test 3

# Locate the sales dataset
sales_path = project_root / "data" / "tableau" / "retail_sales_tableau.csv"

# Load the dataset
sales_df = pd.read_csv(sales_path)

# Display dataset information
print("SALES DATASET VALIDATION")
print("-" * 45)

print(f"Dataset shape: {sales_df.shape}")

print("\nColumn names:")
print(sales_df.columns.tolist())

print("\nFirst five records:")
display(sales_df.head())


SALES DATASET VALIDATION
---------------------------------------------
Dataset shape: (524878, 17)

Column names:
['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country', 'TransactionType', 'Revenue', 'InvoiceYear', 'InvoiceMonth', 'YearMonth', 'InvoiceDay', 'DayOfWeek', 'InvoiceHour', 'CustomerID_Flagged']

First five records:


C:\Users\Joy\AppData\Local\Temp\ipykernel_42408\2568704676.py:7: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  sales_df = pd.read_csv(sales_path)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TransactionType,Revenue,InvoiceYear,InvoiceMonth,YearMonth,InvoiceDay,DayOfWeek,InvoiceHour,CustomerID_Flagged
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,Valid Sale,15.30,2010,12,2010-12,1,Wednesday,8,False
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,Valid Sale,20.34,2010,12,2010-12,1,Wednesday,8,False
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,Valid Sale,22.00,2010,12,2010-12,1,Wednesday,8,False
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,Valid Sale,20.34,2010,12,2010-12,1,Wednesday,8,False
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,Valid Sale,20.34,2010,12,2010-12,1,Wednesday,8,False


In [14]:

# STEP 16B: Validate sales data before invoice aggregation

print("SALES DATA VALIDATION")
print("-" * 45)

# Check missing values in required columns
print("\nMissing values:")
print(sales_df[["InvoiceNo", "Country", "Revenue"]].isnull().sum())

# Count distinct invoices
print("\nDistinct invoices:")
print(sales_df["InvoiceNo"].nunique())

# Check whether invoices have multiple countries
country_per_invoice = sales_df.groupby("InvoiceNo")["Country"].nunique()

print("\nInvoices associated with multiple countries:")
print((country_per_invoice > 1).sum())

# Inspect geographical categories
print("\nCountries and transaction counts:")
print(sales_df["Country"].value_counts().to_string())

# Check revenue quality
print("\nZero or negative revenue rows:")
print((sales_df["Revenue"] <= 0).sum())

print("\nTransaction types:")
print(sales_df["TransactionType"].value_counts(dropna=False))


SALES DATA VALIDATION
---------------------------------------------

Missing values:
InvoiceNo    0
Country      0
Revenue      0
dtype: int64

Distinct invoices:
19962

Invoices associated with multiple countries:
0

Countries and transaction counts:
Country
United Kingdom          479985
Germany                   9025
France                    8392
EIRE                      7879
Spain                     2479
Netherlands               2359
Belgium                   2031
Switzerland               1958
Portugal                  1492
Australia                 1181
Norway                    1071
Italy                      758
Channel Islands            747
Finland                    685
Cyprus                     603
Sweden                     450
Unspecified                442
Austria                    398
Denmark                    380
Poland                     330
Japan                      321
Israel                     292
Hong Kong                  280
Singapore                  

In [15]:

# STEP 16C: Investigate invoice number consistency

print("INVOICE NUMBER VALIDATION")
print("-" * 45)

# Check invoice number data types
print("\nInvoiceNo data type:")
print(sales_df["InvoiceNo"].dtype)

# Identify invoice numbers containing non-numeric characters
invoice_numbers = sales_df["InvoiceNo"].astype(str).str.strip()

non_numeric_invoices = sales_df.loc[
    ~invoice_numbers.str.fullmatch(r"\d+"),
    ["InvoiceNo", "Country", "Revenue", "TransactionType"]
]

print("\nNon-numeric invoice records:")
print(non_numeric_invoices.to_string(index=False))

# Compare invoice counts after numeric conversion
numeric_invoices = pd.to_numeric(invoice_numbers, errors="coerce")

print("\nDistinct original invoices:")
print(invoice_numbers.nunique())

print("\nDistinct numeric invoices:")
print(numeric_invoices.nunique())

print("\nInvoices that cannot be converted to numbers:")
print(numeric_invoices.isna().sum())


INVOICE NUMBER VALIDATION
---------------------------------------------

InvoiceNo data type:
object

Non-numeric invoice records:
InvoiceNo        Country  Revenue TransactionType
  A563185 United Kingdom 11062.06      Valid Sale

Distinct original invoices:
19960

Distinct numeric invoices:
19959

Invoices that cannot be converted to numbers:
1


In [16]:

# STEP 16D: Identify invoice numbers with whitespace

original_invoices = sales_df["InvoiceNo"].astype(str)
clean_invoices = original_invoices.str.strip()

# Find records whose invoice numbers contain surrounding whitespace
whitespace_records = sales_df.loc[
    original_invoices != clean_invoices,
    ["InvoiceNo", "Country", "Revenue"]
].copy()

print("INVOICE WHITESPACE INVESTIGATION")
print("-" * 45)

print("\nRecords containing surrounding whitespace:")
print(len(whitespace_records))

print("\nAffected invoice identifiers:")
print(
    whitespace_records["InvoiceNo"]
    .drop_duplicates()
    .map(repr)
    .to_string(index=False)
)

print("\nDistinct invoices before cleaning:")
print(original_invoices.nunique())

print("\nDistinct invoices after cleaning:")
print(clean_invoices.nunique())


INVOICE WHITESPACE INVESTIGATION
---------------------------------------------

Records containing surrounding whitespace:
0

Affected invoice identifiers:
Series([], )

Distinct invoices before cleaning:
19960

Distinct invoices after cleaning:
19960


In [17]:

# STEP 17: Aggregate sales to invoice level

# Create a working copy
order_source = sales_df.copy()

# Standardise invoice identifiers
order_source["InvoiceNo"] = (
    order_source["InvoiceNo"].astype(str).str.strip()
)

# Aggregate revenue for each invoice and country
order_df = (
    order_source
    .groupby(["InvoiceNo", "Country"], as_index=False)
    .agg(OrderValue=("Revenue", "sum"))
)

# Classify geographical markets
order_df["Market"] = np.where(
    order_df["Country"] == "United Kingdom",
    "UK",
    "International"
)

print("INVOICE-LEVEL DATA VALIDATION")
print("-" * 45)

print(f"Total orders: {len(order_df):,}")
print(f"Unique invoices: {order_df['InvoiceNo'].nunique():,}")

print("\nOrders by market:")
print(order_df["Market"].value_counts())

print("\nRevenue by market:")
print(order_df.groupby("Market")["OrderValue"].sum())

print("\nTotal aggregated revenue:")
print(f"£{order_df['OrderValue'].sum():,.2f}")

print("\nFirst five orders:")
display(order_df.head())


INVOICE-LEVEL DATA VALIDATION
---------------------------------------------
Total orders: 19,960
Unique invoices: 19,960

Orders by market:
Market
UK               18019
International     1941
Name: count, dtype: int64

Revenue by market:
Market
International    1640366.710
UK               9001744.094
Name: OrderValue, dtype: float64

Total aggregated revenue:
£10,642,110.80

First five orders:


,InvoiceNo,Country,OrderValue,Market
0,536365,United Kingdom,139.12,UK
1,536366,United Kingdom,22.20,UK
2,536367,United Kingdom,278.73,UK
3,536368,United Kingdom,70.05,UK
4,536369,United Kingdom,17.85,UK


In [18]:

# STEP 18: Validate geographical classifications

print("COUNTRY-LEVEL ORDER DISTRIBUTION")
print("-" * 50)

country_summary = (
    order_df.groupby("Country")
    .agg(
        TotalOrders=("InvoiceNo", "nunique"),
        TotalRevenue=("OrderValue", "sum")
    )
    .sort_values("TotalOrders", ascending=False)
)

display(country_summary)

print("\nPOTENTIALLY AMBIGUOUS COUNTRY LABELS")
print("-" * 50)

ambiguous_labels = [
    "Unspecified",
    "European Community",
    "Unknown"
]

ambiguous_orders = order_df[
    order_df["Country"].isin(ambiguous_labels)
]

print(f"Orders with ambiguous labels: {len(ambiguous_orders):,}")
print(
    f"Revenue from ambiguous labels: "
    f"£{ambiguous_orders['OrderValue'].sum():,.2f}"
)

display(
    ambiguous_orders.groupby("Country")
    .agg(
        TotalOrders=("InvoiceNo", "nunique"),
        TotalRevenue=("OrderValue", "sum")
    )
)


COUNTRY-LEVEL ORDER DISTRIBUTION
--------------------------------------------------


,TotalOrders,TotalRevenue
Country,,
United Kingdom,18019,9001744.094
Germany,457,228678.400
France,392,209625.370
EIRE,288,283140.520
Belgium,98,41196.340
Netherlands,94,285446.340
Spain,90,61558.560
Portugal,58,33683.050
Australia,57,138453.810



POTENTIALLY AMBIGUOUS COUNTRY LABELS
--------------------------------------------------
Orders with ambiguous labels: 17
Revenue from ambiguous labels: £6,041.19


,TotalOrders,TotalRevenue
Country,,
European Community,4,1300.25
Unspecified,13,4740.94


In [20]:
# STEP 19: Prepare UK and international order groups

# Exclude ambiguous geographical labels
ambiguous_countries = [
    "Unspecified",
    "European Community"
]

market_test_data = order_df[
    ~order_df["Country"].isin(ambiguous_countries)
].copy()

# Separate UK and international order values
uk_order_values = market_test_data.loc[
    market_test_data["Market"] == "UK",
    "OrderValue"
]

international_order_values = market_test_data.loc[
    market_test_data["Market"] == "International",
    "OrderValue"
]

print("UK VS INTERNATIONAL ORDER GROUPS")
print("-" * 45)

print(f"UK orders: {len(uk_order_values):,}")
print(f"International orders: {len(international_order_values):,}")
print(f"Total eligible orders: {len(market_test_data):,}")

print(
    f"Excluded ambiguous orders: "
    f"{len(order_df) - len(market_test_data):,}"
)

print("\nAVERAGE ORDER VALUES")
print(f"UK: £{uk_order_values.mean():,.2f}")
print(f"International: £{international_order_values.mean():,.2f}")

UK VS INTERNATIONAL ORDER GROUPS
---------------------------------------------
UK orders: 18,019
International orders: 1,924
Total eligible orders: 19,943
Excluded ambiguous orders: 17

AVERAGE ORDER VALUES
UK: £499.57
International: £849.44


In [21]:
# STEP 20: Descriptive statistics and distribution analysis

print("UK ORDER VALUE STATISTICS")
print("-" * 45)
print(uk_order_values.describe())

print("\nINTERNATIONAL ORDER VALUE STATISTICS")
print("-" * 45)
print(international_order_values.describe())

print("\nDISTRIBUTION SKEWNESS")
print(f"UK: {uk_order_values.skew():.2f}")
print(f"International: {international_order_values.skew():.2f}")

print("\nMEDIAN ORDER VALUES")
print(f"UK: £{uk_order_values.median():,.2f}")
print(
    f"International: "
    f"£{international_order_values.median():,.2f}"
)

print("\nSTANDARD DEVIATIONS")
print(f"UK: £{uk_order_values.std():,.2f}")
print(
    f"International: "
    f"£{international_order_values.std():,.2f}"
)

UK ORDER VALUE STATISTICS
---------------------------------------------
count     18019.000000
mean        499.569571
std        1781.520653
min           0.380000
25%         147.060000
50%         299.950000
75%         465.315000
max      168469.600000
Name: OrderValue, dtype: float64

INTERNATIONAL ORDER VALUE STATISTICS
---------------------------------------------
count     1924.000000
mean       849.441538
std       1746.775804
min          1.450000
25%        228.862500
50%        427.530000
75%        793.730000
max      22775.930000
Name: OrderValue, dtype: float64

DISTRIBUTION SKEWNESS
UK: 55.78
International: 6.91

MEDIAN ORDER VALUES
UK: £299.95
International: £427.53

STANDARD DEVIATIONS
UK: £1,781.52
International: £1,746.78


In [22]:
# STEP 21: Welch's independent-samples t-test
# Hypothesis Test 3: UK vs International Order Values

t_statistic_market, p_value_market = stats.ttest_ind(
    uk_order_values,
    international_order_values,
    equal_var=False
)

print("WELCH'S INDEPENDENT-SAMPLES T-TEST")
print("-" * 45)

print(f"T-statistic: {t_statistic_market:.4f}")
print(f"P-value: {p_value_market:.6g}")
print(f"Significance level: {alpha}")

print("\nHYPOTHESIS TEST DECISION")

if p_value_market < alpha:
    print("Reject the null hypothesis (H0).")
    print(
        "There is a statistically significant difference "
        "in mean order value between UK and international orders."
    )
else:
    print("Fail to reject the null hypothesis (H0).")
    print(
        "There is insufficient statistical evidence "
        "of a difference in mean order value."
    )

WELCH'S INDEPENDENT-SAMPLES T-TEST
---------------------------------------------
T-statistic: -8.3350
P-value: 1.2954e-16
Significance level: 0.05

HYPOTHESIS TEST DECISION
Reject the null hypothesis (H0).
There is a statistically significant difference in mean order value between UK and international orders.


In [23]:
# STEP 22: Mann-Whitney U test
# Complementary analysis for UK vs International orders

u_statistic_market, mw_p_value_market = stats.mannwhitneyu(
    uk_order_values,
    international_order_values,
    alternative="two-sided"
)

print("MANN-WHITNEY U TEST")
print("-" * 45)

print(f"U-statistic: {u_statistic_market:,.2f}")
print(f"P-value: {mw_p_value_market:.6g}")
print(f"Significance level: {alpha}")

print("\nHYPOTHESIS TEST DECISION")

if mw_p_value_market < alpha:
    print("Reject the null hypothesis (H0).")
    print(
        "There is a statistically significant difference "
        "in the order value distributions between "
        "UK and international orders."
    )
else:
    print("Fail to reject the null hypothesis (H0).")
    print(
        "There is insufficient statistical evidence "
        "of a difference in order value distributions."
    )

MANN-WHITNEY U TEST
---------------------------------------------
U-statistic: 12,685,442.50
P-value: 1.46755e-83
Significance level: 0.05

HYPOTHESIS TEST DECISION
Reject the null hypothesis (H0).
There is a statistically significant difference in the order value distributions between UK and international orders.


### Hypothesis Test 3: Results and Business Interpretation

**Research Question:** Is there a statistically significant difference in order values between UK and international orders?

**Descriptive Findings**

- UK orders (n = 18,019): Mean = £499.57; median = £299.95.
- International orders (n = 1,924): Mean = £849.44; median = £427.53.
- International orders have a mean value approximately £349.87 higher than UK orders.
- Seventeen orders with ambiguous geographical labels were excluded from this comparison.

**Statistical Results**

- **Welch's t-test:** t = -8.3350, p = 1.2954 × 10⁻¹⁶.
- **Mann–Whitney U test:** U = 12,685,442.50, p = 1.46755 × 10⁻⁸³.
- **Significance level:** α = 0.05.

**Conclusion**

Both tests reject their respective null hypotheses. Welch's t-test identifies a statistically significant difference in mean order values, while the Mann–Whitney U test identifies a statistically significant difference in order value distributions. International orders exhibit higher mean and median values than UK orders.

**Business Implications**

The retailer should investigate opportunities to expand profitable international sales, particularly in markets with strong order values. International marketing, customer segmentation and shipping strategies should be evaluated alongside fulfilment costs and profitability.

**Limitations**

Both distributions are heavily right-skewed, particularly UK orders, which include extreme values. The results are observational and do not establish that geographical location causes differences in spending. Differences in product mix, customer composition, shipping charges and purchasing behaviour may contribute to the findings. The international group also combines multiple countries with different market characteristics.